# Strobl controlled benchmark — dataset QC

> Synthetic methodological benchmark only. This dataset has no clinical validity and contains no patient data.

This lightweight notebook loads the generated pilot (or smoke fallback), reruns the reusable invariants, and displays representative spatial states and trajectory summaries.

In [ ]:
from pathlib import Path
import json
import h5py
import matplotlib.pyplot as plt
import numpy as np

from koopman_control.data.qc_strobl import inspect_dataset
from koopman_control.paths import strobl_dataset_path

pilot = strobl_dataset_path("pilot")
smoke = strobl_dataset_path("smoke")
DATASET = pilot if pilot.exists() else smoke
print(f"QC dataset: {DATASET}")
report = inspect_dataset(DATASET)
report

In [ ]:
with h5py.File(DATASET, "r") as h5:
    print("Schema:", h5.attrs["schema_version"])
    print("Upstream commit:", h5.attrs["upstream_commit"])
    print("Action alignment:", h5.attrs["action_alignment"])
    print("Episodes:", len(h5["episodes"]))
    for horizon in (1, 5, 10, 25):
        counts = {
            split: len(h5[f"transition_index/H{horizon}/{split}/t"])
            for split in ("train", "val", "test")
        }
        print(f"H={horizon}:", counts)

summary_path = DATASET.parent / "qc_summary.json"
if summary_path.exists():
    print(json.dumps(json.loads(summary_path.read_text()), indent=2))

In [ ]:
image = plt.imread(DATASET.parent / "qc_examples.png")
fig, ax = plt.subplots(figsize=(12, 16))
ax.imshow(image)
ax.axis("off")
ax.set_title("Representative initial/final grids and trajectories")
plt.show()